In [11]:
import os
import re
import csv
import pandas as pd
from numpy import trapezoid


def parse_graph_stats(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    name_match = re.search(r'графе (.+?):', content)
    graph_name = name_match.group(1).strip() if name_match else os.path.basename(filepath)

    vertices = int(re.search(r'Количество вершин: (\d+)', content).group(1))
    edges = int(re.search(r'Количество рёбер: (\d+)', content).group(1))
    density = float(re.search(r'Плотность графа: ([\d.]+)', content).group(1))
    count_of_wcc = int(re.search(r'Количество WCC: (\d+)', content).group(1))
    v_largest_wcc = float(re.search(r'Доля вершин в наибольшей WCC: ([\d.]+)', content).group(1))
    count_of_scc = int(re.search(r'Количество SCC: (\d+)', content).group(1))
    v_largest_scc = float(re.search(r'Доля вершин в наибольшей SCC: ([\d.]+)', content).group(1))
    diameter = int(re.search(r'Точный диаметр графа: (\d+)', content).group(1))
    radius = int(re.search(r'Точный радиус графа: (\d+)', content).group(1))
    diameter_ds = float(re.search(r'Диаметр наибольшей WCC \(The Double Sweep\): ([\d.]+)', content).group(1))
    percentile_ds_500 = float(re.search(r'90 процентиль расстояний \(random 500\): ([\d.]+)', content).group(1))
    percentile_ds_1000 = float(re.search(r'90 процентиль расстояний \(random 1000\): ([\d.]+)', content).group(1))
    diameter_s_500 = float(re.search(r'Диаметр наибольшей WCC \(Snowball-500 \+ Double Sweep\): ([\d.]+)', content).group(1))
    diameter_s_1000 = float(re.search(r'Диаметр наибольшей WCC \(Snowball-1000 \+ Double Sweep\): ([\d.]+)', content).group(1))
    percentile_s_500 = float(re.search(r'90 процентиль расстояний \(Snowball-500\): ([\d.]+)', content).group(1))
    percentile_s_1000 = float(re.search(r'90 процентиль расстояний \(Snowball-1000\): ([\d.]+)', content).group(1))
    cluster_coeff = float(re.search(r'Кластерный коэффициент вершины номера команды \(5\): ([\d.]+)', content).group(1))
    triangles = int(re.search(r'Количество треугольников: (\d+)', content).group(1))
    avg_clustering = float(re.search(r'Средний коэффициент кластеризации: ([\d.]+)', content).group(1))
    global_clustering = float(re.search(r'Глобальный коэффициент кластеризации: ([\d.]+)', content).group(1))
    avg_clustering_wcc = float(re.search(r'Средний коэффициент кластеризации \(largest WCC\): ([\d.]+)', content).group(1))
    min_deg = float(re.search(r'Минимальная степень узлов: (\d+)', content).group(1))
    avg_deg = float(re.search(r'Средняя степень узлов: ([\d.]+)', content).group(1))
    max_deg = float(re.search(r'Максимальная степень узлов: (\d+)', content).group(1))

    def extract_block(header):
        match = re.search(header + r':\s*((?:\n\t+x = [^\n]+)+)', content)
        return match.group(1) if match else ''

    def extract_removal_data(block):
        data = {}
        for match in re.findall(r'x = ([\d.]+)%: доля вершин в наибольшей WCC: ([\d.]+)', block):
            x = float(match[0])
            y = float(match[1])
            data[x] = y
        return data

    block_random = extract_block(r'Удаление случайных узлов')
    block_degree = extract_block(r'Удаление узлов наибольшей степени')

    data_random = extract_removal_data(block_random)
    data_degree = extract_removal_data(block_degree)

    def compute_auc(data):
        if not data: return None
        x = sorted(data.keys())
        y = [data[k] for k in x]
        return trapezoid(y, x)

    auc_random = compute_auc(data_random)
    auc_degree = compute_auc(data_degree)

    return {
        'graph': graph_name,
        'vertices': vertices,
        'edges': edges,
        'density': density,
        'count_of_wcc': count_of_wcc,
        'v_largest_wcc': v_largest_wcc,
        'count_of_scc': count_of_scc,
        'v_largest_scc': v_largest_scc,
        'diameter': diameter,
        'radius': radius,
        'diameter_ds': diameter_ds,
        'percentile_ds_500': percentile_ds_500,
        'percentile_ds_1000': percentile_ds_1000,
        'diameter_s_500': diameter_s_500,
        'diameter_s_1000': diameter_s_1000,
        'percentile_s_500': percentile_s_500,
        'percentile_s_1000': percentile_s_1000,
        'cluster_coeff': cluster_coeff,
        'triangles': triangles,
        'avg_clustering': avg_clustering,
        'global_clustering': global_clustering,
        'avg_clustering_wcc': avg_clustering_wcc,
        'min_deg': min_deg,
        'avg_deg': avg_deg,
        'max_deg': max_deg,
        'auc_random': auc_random,
        'auc_degree': auc_degree
    }


input_dir = r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\output\tests'
output_csv = os.path.join(r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization', 'graph_tests.csv')

results = []

for file in os.listdir(input_dir):
    if file.endswith('.txt'):
        full_path = os.path.join(input_dir, file)
        stats = parse_graph_stats(full_path)
        results.append(stats)


with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        'graph', 'vertices', 'edges', 'density', 'count_of_wcc', 'v_largest_wcc', 'diameter', 'radius',
        'count_of_scc', 'v_largest_scc', 'diameter_ds', 'percentile_ds_500', 'percentile_ds_1000', 'diameter_s_500', 'diameter_s_1000',
        'percentile_s_500', 'percentile_s_1000', 'cluster_coeff', 'triangles', 'avg_clustering', 'global_clustering', 'avg_clustering_wcc',
        'min_deg', 'avg_deg', 'max_deg', 'auc_random', 'auc_degree'
    ])
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print("CSV сохранён:", output_csv)

df = pd.read_csv(output_csv)
display(df)


CSV сохранён: C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization\graph_tests.csv


,graph,vertices,edges,density,count_of_wcc,v_largest_wcc,diameter,radius,count_of_scc,v_largest_scc,...,cluster_coeff,triangles,avg_clustering,global_clustering,avg_clustering_wcc,min_deg,avg_deg,max_deg,auc_random,auc_degree
0,digraph_1,1000,5062,0.010108,1,1.000000,5,4,17,0.984000,...,0.02,169,0.0104,0.0100,0.0104,3.0,10.10,21.0,74.62400,56.82850
1,digraph_2,1000,4053,0.008086,1,1.000000,6,5,45,0.956000,...,0.00,86,0.0091,0.0080,0.0091,1.0,8.08,17.0,65.78200,47.33125
2,digraph_3,1000,2947,0.005894,1,1.000000,8,5,107,0.894000,...,0.00,34,0.0060,0.0059,0.0060,1.0,5.89,16.0,63.48425,37.11000
3,graph_0,34,78,0.139037,1,1.000000,5,3,34,0.029412,...,0.67,45,0.5706,0.2557,0.5706,1.0,4.59,17.0,60.52350,14.39150
4,graph_1,962,18812,0.040697,1,1.000000,6,4,962,0.001040,...,0.15,97137,0.3184,0.2207,0.3184,1.0,39.11,313.0,75.87925,50.70900
5,graph_2,3075,124610,0.026365,4,0.998049,7,1,3075,0.000325,...,0.25,1119231,0.2816,0.2108,0.2822,1.0,81.05,473.0,78.00675,63.31850
6,graph_3,864,995,0.002669,28,0.906250,20,1,864,0.001157,...,0.00,2,0.0018,0.0030,0.0020,1.0,2.30,8.0,24.88725,2.05925
7,graph_4,955,1447,0.003176,5,0.991623,14,1,955,0.001047,...,0.00,3,0.0022,0.0022,0.0022,1.0,3.03,10.0,38.36475,11.17150
8,graph_5,991,2490,0.005076,1,1.000000,9,6,991,0.001009,...,0.00,24,0.0055,0.0057,0.0055,1.0,5.03,14.0,59.26450,27.83925
9,graph_6,999,3737,0.007496,1,1.000000,7,5,999,0.001001,...,0.00,71,0.0079,0.0077,0.0079,1.0,7.48,19.0,68.05400,46.78525


In [5]:
selected_columns = ['graph', 'vertices', 'edges', 'density', 'count_of_wcc', 'v_largest_wcc', 'count_of_scc', 'v_largest_scc']
df[selected_columns]

,graph,vertices,edges,density,count_of_wcc,v_largest_wcc,count_of_scc,v_largest_scc
0,digraph_1,1000,5062,0.010108,1,1.000000,17,0.984000
1,digraph_2,1000,4053,0.008086,1,1.000000,45,0.956000
2,digraph_3,1000,2947,0.005894,1,1.000000,107,0.894000
3,graph_0,34,78,0.139037,1,1.000000,34,0.029412
4,graph_1,962,18812,0.040697,1,1.000000,962,0.001040
5,graph_2,3075,124610,0.026365,4,0.998049,3075,0.000325
6,graph_3,864,995,0.002669,28,0.906250,864,0.001157
7,graph_4,955,1447,0.003176,5,0.991623,955,0.001047
8,graph_5,991,2490,0.005076,1,1.000000,991,0.001009
9,graph_6,999,3737,0.007496,1,1.000000,999,0.001001


In [6]:
selected_columns = ['graph', 'diameter_ds', 'percentile_ds_500', 'percentile_ds_1000', 'diameter_s_500', 'diameter_s_1000', 'percentile_s_500', 'percentile_s_1000']
df[selected_columns]

,graph,diameter_ds,percentile_ds_500,percentile_ds_1000,diameter_s_500,diameter_s_1000,percentile_s_500,percentile_s_1000
0,digraph_1,4.86,4.00,4.00,6.16,8.68,6.12,7.24
1,digraph_2,5.40,4.00,4.00,6.72,10.06,6.58,8.06
2,digraph_3,7.16,5.00,5.00,8.04,12.12,7.68,9.14
3,graph_0,5.00,4.00,4.00,6.58,6.62,5.08,5.11
4,graph_1,5.68,3.00,3.00,4.24,8.44,4.22,5.48
5,graph_2,5.92,3.00,3.00,4.08,4.22,4.06,4.18
6,graph_3,19.14,12.00,12.00,17.78,28.38,15.80,19.30
7,graph_4,12.98,8.07,8.00,12.30,19.52,11.62,14.00
8,graph_5,8.10,5.92,6.00,8.62,13.22,8.36,10.00
9,graph_6,6.04,4.33,4.29,7.04,10.30,6.88,8.18


In [7]:
selected_columns = ['graph', 'triangles', 'avg_clustering', 'global_clustering']
df[selected_columns]

,graph,triangles,avg_clustering,global_clustering
0,digraph_1,169,0.0104,0.0100
1,digraph_2,86,0.0091,0.0080
2,digraph_3,34,0.0060,0.0059
3,graph_0,45,0.5706,0.2557
4,graph_1,97137,0.3184,0.2207
5,graph_2,1119231,0.2816,0.2108
6,graph_3,2,0.0018,0.0030
7,graph_4,3,0.0022,0.0022
8,graph_5,24,0.0055,0.0057
9,graph_6,71,0.0079,0.0077


In [8]:
selected_columns = ['graph', 'avg_clustering_wcc']
df[selected_columns]

,graph,avg_clustering_wcc
0,digraph_1,0.0104
1,digraph_2,0.0091
2,digraph_3,0.0060
3,graph_0,0.5706
4,graph_1,0.3184
5,graph_2,0.2822
6,graph_3,0.0020
7,graph_4,0.0022
8,graph_5,0.0055
9,graph_6,0.0079


In [10]:
selected_columns = ['graph', 'min_deg', 'avg_deg', 'max_deg']
df[selected_columns]

,graph,min_deg,avg_deg,max_deg
0,digraph_1,3.0,10.10,21.0
1,digraph_2,1.0,8.08,17.0
2,digraph_3,1.0,5.89,16.0
3,graph_0,1.0,4.59,17.0
4,graph_1,1.0,39.11,313.0
5,graph_2,1.0,81.05,473.0
6,graph_3,1.0,2.30,8.0
7,graph_4,1.0,3.03,10.0
8,graph_5,1.0,5.03,14.0
9,graph_6,1.0,7.48,19.0


In [12]:
selected_columns = ['graph', 'diameter', 'radius', 'cluster_coeff']
df[selected_columns]

,graph,diameter,radius,cluster_coeff
0,digraph_1,5,4,0.02
1,digraph_2,6,5,0.00
2,digraph_3,8,5,0.00
3,graph_0,5,3,0.67
4,graph_1,6,4,0.15
5,graph_2,7,1,0.25
6,graph_3,20,1,0.00
7,graph_4,14,1,0.00
8,graph_5,9,6,0.00
9,graph_6,7,5,0.00
